In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned dataset from the project data folder
cleaned_data_candidates = [
    Path('cleaned_dataset_supplychain.csv'),
    Path('..') / 'data' / 'processed' / 'cleaned_dataset_supplychain.csv',
    Path('data') / 'processed' / 'cleaned_dataset_supplychain.csv',
]
cleaned_data_path = next((path for path in cleaned_data_candidates if path.exists()), None)
if cleaned_data_path is None:
    raise FileNotFoundError('Could not find cleaned_dataset_supplychain.csv in the notebook folder or data/processed.')

df = pd.read_csv(cleaned_data_path)

# Convert date column
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Create a clean date-only column
df['order_date'] = df['order date (DateOrders)'].dt.normalize()

print("Loaded shape:", df.shape)
print("Date range:", df['order_date'].min(), "→", df['order_date'].max())
print("Columns:", df.columns.tolist())

Loaded shape: (180519, 54)
Date range: 2015-01-01 00:00:00 → 2018-01-31 00:00:00
Columns: ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'P

In [2]:
# See exactly where files will be saved
print("All files will save to:", os.getcwd())
# This is your notebooks/ folder — all 3 output files go here

All files will save to: e:\supply-chain-optimization\notebooks


In [3]:
# Group all orders by date — sum up total units sold per day
demand_daily = df.groupby('order_date').agg(
    y           = ('Order Item Quantity', 'sum'),
    total_sales = ('Sales', 'sum'),
    order_count = ('Order Id', 'count')
).reset_index()

demand_daily = demand_daily.rename(columns={'order_date': 'ds'})
demand_daily = demand_daily.sort_values('ds').reset_index(drop=True)

print("Daily demand rows:", len(demand_daily))
print(demand_daily.head())

Daily demand rows: 1127
          ds    y   total_sales  order_count
0 2015-01-01  355  32806.090690          168
1 2015-01-02  354  29818.210575          154
2 2015-01-03  392  36348.710648          179
3 2015-01-04  410  35738.970669          191
4 2015-01-05  373  31067.910603          160


In [4]:
demand_daily = demand_daily.copy()

# Time features
demand_daily['day_of_week'] = demand_daily['ds'].dt.dayofweek
demand_daily['month']       = demand_daily['ds'].dt.month
demand_daily['quarter']     = demand_daily['ds'].dt.quarter

# Week number — safe version for Python 3.13
demand_daily['week'] = demand_daily['ds'].dt.isocalendar().week.astype(int)

# Lag features — what was demand 7/14/21/30 days ago
for lag in [7, 14, 21, 30]:
    demand_daily[f'lag_{lag}'] = demand_daily['y'].shift(lag)

# Rolling averages — smooth trend over past days
demand_daily['rolling_7d_avg']  = demand_daily['y'].shift(1).rolling(7).mean()
demand_daily['rolling_14d_avg'] = demand_daily['y'].shift(1).rolling(14).mean()
demand_daily['rolling_30d_avg'] = demand_daily['y'].shift(1).rolling(30).mean()

# Drop first 30 rows where lag values are NaN
demand_features = demand_daily.dropna().reset_index(drop=True)

print("Shape after lag features:", demand_features.shape)
print("Columns:", demand_features.columns.tolist())
print(demand_features.head(3))

Shape after lag features: (1097, 15)
Columns: ['ds', 'y', 'total_sales', 'order_count', 'day_of_week', 'month', 'quarter', 'week', 'lag_7', 'lag_14', 'lag_21', 'lag_30', 'rolling_7d_avg', 'rolling_14d_avg', 'rolling_30d_avg']
          ds    y   total_sales  order_count  day_of_week  month  quarter  \
0 2015-01-31  402  35514.030669          187            5      1        1   
1 2015-02-01  383  33479.750717          181            6      2        1   
2 2015-02-02  327  31396.610598          160            0      2        1   

   week  lag_7  lag_14  lag_21  lag_30  rolling_7d_avg  rolling_14d_avg  \
0     5  384.0   349.0   384.0   355.0      378.000000       384.571429   
1     5  365.0   407.0   363.0   354.0      380.571429       388.357143   
2     6  407.0   346.0   493.0   392.0      383.142857       386.642857   

   rolling_30d_avg  
0       381.733333  
1       383.300000  
2       384.266667  


In [5]:
demand_features.to_csv("demand_features.csv", index=False)
print("✅ Saved: demand_features.csv")
print("Rows:", len(demand_features), "| Columns:", len(demand_features.columns))

✅ Saved: demand_features.csv
Rows: 1097 | Columns: 15


In [10]:
from pathlib import Path
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Load your cleaned file
cleaned_data_candidates = [
    Path('cleaned_dataset_supplychain.csv'),
    Path('..') / 'data' / 'processed' / 'cleaned_dataset_supplychain.csv',
    Path('data') / 'processed' / 'cleaned_dataset_supplychain.csv',
]
cleaned_data_path = next((path for path in cleaned_data_candidates if path.exists()), None)
if cleaned_data_path is None:
    raise FileNotFoundError('Could not find cleaned_dataset_supplychain.csv in the notebook folder or data/processed.')

df = pd.read_csv(cleaned_data_path)
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Encode shipping mode
ship_mode_map = {
    'First Class'   : 0,
    'Second Class'  : 1,
    'Standard Class': 2,
    'Same Day'      : 3
}

# Encode market
market_map = {
    'Africa'       : 0,
    'Europe'       : 1,
    'LATAM'        : 2,
    'Pacific Asia' : 3,
    'USCA'         : 4
}

# Encode customer segment
segment_map = {
    'Consumer'   : 0,
    'Corporate'  : 1,
    'Home Office': 2
}

df['shipping_mode_enc'] = df['Shipping Mode'].map(ship_mode_map).fillna(2)
df['market_enc']        = df['Market'].map(market_map).fillna(0)
df['segment_enc']       = df['Customer Segment'].map(segment_map).fillna(0)

# Time features
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_dow']   = df['order date (DateOrders)'].dt.dayofweek
df['order_week']  = df['order date (DateOrders)'].dt.isocalendar().week.astype(int)

# Select only needed columns
delay_features = df[[
    'shipping_mode_enc',
    'market_enc',
    'segment_enc',
    'Days for shipment (scheduled)',
    'Order Item Quantity',
    'Order Item Discount Rate',
    'Order Item Profit Ratio',
    'Sales',
    'order_month',
    'order_dow',
    'order_week',
    'Late_delivery_risk'
]].copy()

delay_features = delay_features.dropna().reset_index(drop=True)

print("Shape:", delay_features.shape)
print("Target breakdown:")
print(delay_features['Late_delivery_risk'].value_counts())
print(f"Late delivery rate: {delay_features['Late_delivery_risk'].mean()*100:.1f}%")

# Save
delay_features.to_csv("delay_features.csv", index=False)
print("\n✅ Saved: delay_features.csv")

Shape: (180519, 12)
Target breakdown:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
Late delivery rate: 54.8%

✅ Saved: delay_features.csv


In [4]:
from pathlib import Path
import pandas as pd

if 'df' not in globals():
    data_path_candidates = [
        Path('cleaned_dataset_supplychain.csv'),
        Path('..') / 'cleaned_dataset_supplychain.csv',
        Path('data') / 'processed' / 'cleaned_dataset_supplychain.csv',
    ]
    data_path = next((path for path in data_path_candidates if path.exists()), None)
    if data_path is None:
        raise FileNotFoundError('Could not find cleaned_dataset_supplychain.csv in the notebook folder or project data directory.')
    df = pd.read_csv(data_path)

if 'order_date' not in df.columns:
    df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
    df['order_date'] = df['order date (DateOrders)'].dt.normalize()

latest_date = df['order_date'].max()

product_stats = df.groupby('Product Name').agg(
    total_qty_sold   = ('Order Item Quantity', 'sum'),
    total_revenue    = ('Sales', 'sum'),
    order_count      = ('Order Id', 'count'),
    avg_unit_price   = ('Product Price', 'mean'),
    first_order_date = ('order_date', 'min'),
    last_order_date  = ('order_date', 'max')
).reset_index()

# Days since product last sold
product_stats['days_since_last_sale'] = (
    latest_date - pd.to_datetime(product_stats['last_order_date'])
).dt.days

# How long product has been active
product_stats['active_days'] = (
    pd.to_datetime(product_stats['last_order_date']) -
    pd.to_datetime(product_stats['first_order_date'])
).dt.days + 1

# How fast it sells per week
product_stats['weekly_sales_velocity'] = (
    product_stats['total_qty_sold'] /
    (product_stats['active_days'] / 7)
).round(3)

print("Products found:", len(product_stats))
print(product_stats.head())

Products found: 118
                        Product Name  total_qty_sold  total_revenue  \
0                 Adult dog supplies             492   41524.800753   
1                       Baby sweater             207   12229.560379   
2            Bag Boy Beverage Holder             845   21116.549776   
3             Bag Boy M330 Push Cart             208   16637.919929   
4  Bowflex SelectTech 1090 Dumbbells              10    5999.899902   

   order_count  avg_unit_price first_order_date last_order_date  \
0          492       84.400002       2017-11-29      2018-01-10   
1          207       59.080002       2017-10-07      2017-12-19   
2          279       24.990000       2015-01-02      2017-04-18   
3           69       79.989998       2017-04-27      2017-09-11   
4           10      599.989990       2017-09-29      2017-10-01   

   days_since_last_sale  active_days  weekly_sales_velocity  
0                    21           43                 80.093  
1                    43   

In [5]:
# ── ABC: by revenue share ──────────────────────────────────
product_stats = product_stats.sort_values(
    'total_revenue', ascending=False
).reset_index(drop=True)

total_rev = product_stats['total_revenue'].sum()
product_stats['revenue_pct']    = product_stats['total_revenue'] / total_rev * 100
product_stats['revenue_cumsum'] = product_stats['revenue_pct'].cumsum()

def abc_classify(cumsum):
    if cumsum <= 80: return 'A'
    elif cumsum <= 95: return 'B'
    else: return 'C'

product_stats['ABC'] = product_stats['revenue_cumsum'].apply(abc_classify)

# ── XYZ: by demand consistency ─────────────────────────────
df['year_month'] = df['order date (DateOrders)'].dt.to_period('M')

monthly_sales = df.groupby(
    ['Product Name', 'year_month']
)['Order Item Quantity'].sum().unstack(fill_value=0)

cv = (monthly_sales.std(axis=1) / (monthly_sales.mean(axis=1) + 1e-9)).fillna(0)
xyz_df = cv.reset_index()
xyz_df.columns = ['Product Name', 'cv_value']

def xyz_classify(cv_val):
    if cv_val < 0.5:  return 'X'
    elif cv_val < 1.0: return 'Y'
    else: return 'Z'

xyz_df['XYZ'] = xyz_df['cv_value'].apply(xyz_classify)

# Merge and combine
product_stats = product_stats.merge(xyz_df, on='Product Name', how='left')
product_stats['ABC_XYZ'] = product_stats['ABC'] + product_stats['XYZ']

print("ABC distribution:")
print(product_stats['ABC'].value_counts())
print("\nXYZ distribution:")
print(product_stats['XYZ'].value_counts())
print("\nABC_XYZ segments:")
print(product_stats['ABC_XYZ'].value_counts())

ABC distribution:
ABC
C    95
B    16
A     7
Name: count, dtype: int64

XYZ distribution:
XYZ
Z    64
Y    45
X     9
Name: count, dtype: int64

ABC_XYZ segments:
ABC_XYZ
CZ    53
CY    42
BZ    11
AX     7
BY     3
BX     2
Name: count, dtype: int64


In [7]:
product_stats.to_csv("dead_stock_features.csv", index=False)
print("✅ Saved: dead_stock_features.csv")
print("Rows:", len(product_stats), "| Columns:", len(product_stats.columns))

✅ Saved: dead_stock_features.csv
Rows: 118 | Columns: 16


In [9]:
files = [
    "demand_features.csv",
    "delay_features.csv",
    "dead_stock_features.csv"
]

print("=== FEATURE ENGINEERING COMPLETE ===\n")
for f in files:
    if os.path.exists(f):
        temp = pd.read_csv(f)
        print(f"{f}: {temp.shape[0]} rows × {temp.shape[1]} columns")
    else:
        print(f"NOT FOUND: {f}")

=== FEATURE ENGINEERING COMPLETE ===

demand_features.csv: 1097 rows × 15 columns
delay_features.csv: 180519 rows × 12 columns
dead_stock_features.csv: 118 rows × 16 columns
